# JARVIS - Voice Assistant
**Refactored Version**

A Python-based voice assistant with improved security, error handling, and architecture.

- Author: Tumelo
- Improvements: Security fixes, code organization, extensibility

## Setup and Dependencies

In [ ]:
import os
import sys

# Install system dependencies (Colab)
if not os.path.exists('/usr/include/portaudio.h'):
    print('Installing system dependencies...')
    os.system('sudo apt-get update')
    os.system('sudo apt-get install -y portaudio19-dev')
    print('System dependencies installed.')

# Install Python packages
os.system('pip install -q pyttsx3 SpeechRecognition PyAudio')
print('Python packages installed.')

## JARVIS Configuration and Setup

In [ ]:
import datetime
import webbrowser
import urllib.parse
import logging
import math
import ast
import operator as op
import pyttsx3
import speech_recognition as sr

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
CONFIG = {
    'ASSISTANT_NAME': 'JARVIS',
    'TTS_RATE': 170,
    'TTS_VOLUME': 1.0,
    'LISTEN_TIMEOUT': 5,
    'PHRASE_TIME_LIMIT': 5,
    'PAUSE_THRESHOLD': 1.0,
    'DEBUG': False
}

# Text-to-Speech Engine Setup
def initialize_tts_engine():
    """Initialize the text-to-speech engine with fallback support."""
    try:
        engine = pyttsx3.init('espeak')
        engine.setProperty('rate', CONFIG['TTS_RATE'])
        engine.setProperty('volume', CONFIG['TTS_VOLUME'])
        voices = engine.getProperty('voices')
        if voices:
            target_voice_id = next(
                (v.id for v in voices if 'en' in v.id.lower()),
                voices[0].id
            )
            engine.setProperty('voice', target_voice_id)
        logger.info('TTS engine initialized successfully')
        return engine
    except Exception as e:
        logger.warning(f'TTS Engine initialization failed: {e}. Using fallback.')
        return DummyTTSEngine()

class DummyTTSEngine:
    """Fallback engine when TTS is unavailable."""
    def say(self, text):
        print(f'[TTS UNAVAILABLE] {text}')
    
    def runAndWait(self):
        pass
    
    def setProperty(self, key, value):
        pass
    
    def getProperty(self, key):
        return []

engine = initialize_tts_engine()

## Core Functions: Speech I/O and Utilities

In [ ]:
def speak(text):
    """Output text via TTS and print to console.
    
    Args:
        text (str): Text to speak
    """
    print(f'{CONFIG["ASSISTANT_NAME"]}: {text}')
    try:
        engine.say(text)
        engine.runAndWait()
    except Exception as e:
        logger.error(f'Speech error: {e}')

def listen():
    """Listen for voice input and convert to text.
    
    Returns:
        str: Recognized text in lowercase, or empty string on error
    """
    recognizer = sr.Recognizer()
    try:
        with sr.Microphone() as source:
            print('[LISTENING...]')
            recognizer.pause_threshold = CONFIG['PAUSE_THRESHOLD']
            audio = recognizer.listen(
                source,
                timeout=CONFIG['LISTEN_TIMEOUT'],
                phrase_time_limit=CONFIG['PHRASE_TIME_LIMIT']
            )
            user_input = recognizer.recognize_google(
                audio,
                language='en-US'
            ).lower()
            print(f'You: {user_input}')
            logger.debug(f'Recognized: {user_input}')
            return user_input
    except sr.UnknownValueError:
        logger.warning('Speech not recognized')
        speak('Sorry, I did not understand that.')
        return ''
    except sr.RequestError as e:
        logger.error(f'Speech recognition service error: {e}')
        speak('Speech recognition service is unavailable.')
        return ''
    except Exception as e:
        logger.error(f'Listening error: {e}')
        return ''

def tell_time():
    """Tell the current time."""
    now = datetime.datetime.now().strftime('%I:%M %p')
    speak(f'The current time is {now}')

def open_search(query):
    """Open a Google search in the default browser.
    
    Args:
        query (str): Search query
    """
    if not query:
        return
    try:
        encoded_query = urllib.parse.quote(query)
        webbrowser.open(f'https://www.google.com/search?q={encoded_query}')
        logger.info(f'Opened search for: {query}')
    except Exception as e:
        logger.error(f'Browser error: {e}')
        speak('Could not open browser.')


## Safe Math Evaluation

In [ ]:
def safe_eval_math(expression):
    """Safely evaluate mathematical expressions without using eval().
    
    Uses AST parsing for security. Supports basic arithmetic and math functions.
    
    Args:
        expression (str): Mathematical expression (e.g., '2 + 3 * 4')
    
    Returns:
        float: Result of evaluation, or None on error
    """
    ALLOWED_FUNCTIONS = {
        'sqrt': math.sqrt,
        'sin': math.sin,
        'cos': math.cos,
        'tan': math.tan,
        'pi': math.pi,
        'e': math.e,
        'abs': abs,
        'round': round,
    }
    
    ALLOWED_OPERATORS = {
        ast.Add: op.add,
        ast.Sub: op.sub,
        ast.Mult: op.mul,
        ast.Div: op.truediv,
        ast.Pow: op.pow,
        ast.USub: op.neg,
    }
    
    class MathEvaluator(ast.NodeVisitor):
        def visit_BinOp(self, node):
            left = self.visit(node.left)
            right = self.visit(node.right)
            if left is None or right is None:
                return None
            op_type = type(node.op)
            if op_type not in ALLOWED_OPERATORS:
                logger.error(f'Unsupported operator: {op_type}')
                return None
            return ALLOWED_OPERATORS[op_type](left, right)
        
        def visit_UnaryOp(self, node):
            operand = self.visit(node.operand)
            op_type = type(node.op)
            if op_type not in ALLOWED_OPERATORS:
                return None
            return ALLOWED_OPERATORS[op_type](operand)
        
        def visit_Call(self, node):
            if isinstance(node.func, ast.Name):
                func_name = node.func.id
                if func_name not in ALLOWED_FUNCTIONS:
                    logger.error(f'Unsupported function: {func_name}')
                    return None
                args = [self.visit(arg) for arg in node.args]
                if None in args:
                    return None
                return ALLOWED_FUNCTIONS[func_name](*args)
            return None
        
        def visit_Constant(self, node):
            return node.value
        
        def visit_Num(self, node):  # For older Python versions
            return node.n
    
    try:
        tree = ast.parse(expression, mode='eval')
        evaluator = MathEvaluator()
        result = evaluator.visit(tree.body)
        if result is None:
            logger.warning(f'Invalid expression: {expression}')
            return None
        return result
    except SyntaxError:
        logger.error(f'Syntax error in expression: {expression}')
        return None
    except Exception as e:
        logger.error(f'Math evaluation error: {e}')
        return None

def calculate(expression):
    """Process a calculation request.
    
    Args:
        expression (str): Mathematical expression to evaluate
    """
    if not expression:
        speak('No expression provided.')
        return
    
    result = safe_eval_math(expression)
    if result is not None:
        speak(f'The result is {result}')
    else:
        speak('I could not calculate that expression.')


## Command Processing with Registry Pattern

In [ ]:
class CommandRegistry:
    """Registry for managing commands in a scalable way."""
    
    def __init__(self):
        self.commands = {}
    
    def register(self, keywords, handler, description=''):
        """Register a command.
        
        Args:
            keywords (list): Keywords that trigger this command
            handler (callable): Function to handle the command
            description (str): Description of the command
        """
        for keyword in keywords:
            self.commands[keyword] = {'handler': handler, 'description': description}
    
    def get_handler(self, command_text):
        """Get handler for a command.
        
        Args:
            command_text (str): User command
        
        Returns:
            callable: Handler function, or None if not found
        """
        for keyword, command_info in self.commands.items():
            if keyword in command_text:
                return command_info['handler']
        return None

# Initialize command registry
registry = CommandRegistry()

def handle_greet(cmd):
    speak(f'Hello! I am {CONFIG["ASSISTANT_NAME"]}.')

def handle_math(cmd):
    speak('What is the expression?')
    expression = listen()
    calculate(expression)

def handle_search(cmd):
    speak('What should I search for?')
    query = listen()
    if query:
        open_search(query)
        speak('Opening search results.')

def handle_help(cmd):
    help_text = 'I can help you with: time, math, search, and more. Say hello to get started.'
    speak(help_text)

# Register commands
registry.register(['hello', 'hi', 'hey'], handle_greet, 'Greet the assistant')
registry.register(['time', 'what time'], tell_time, 'Get current time')
registry.register(['math', 'calculate'], handle_math, 'Perform calculation')
registry.register(['search', 'google'], handle_search, 'Search on Google')
registry.register(['help', 'commands'], handle_help, 'Show available commands')

def process_command(command):
    """Process a user command.
    
    Args:
        command (str): User command text
    
    Returns:
        bool: True to continue, False to exit
    """
    if not command:
        return True
    
    # Check for exit commands
    if any(word in command for word in ['bye', 'quit', 'exit', 'goodbye']):
        speak('Goodbye!')
        return False
    
    # Look up handler in registry
    handler = registry.get_handler(command)
    if handler:
        try:
            handler(command)
        except Exception as e:
            logger.error(f'Command handler error: {e}')
            speak('An error occurred processing that command.')
    else:
        logger.info(f'Unknown command: {command}')
        speak('Sorry, I did not understand that. Say help for available commands.')
    
    return True


## Main Loop

In [ ]:
def main():
    """Main loop for the JARVIS assistant."""
    try:
        speak(f'{CONFIG["ASSISTANT_NAME"]} system online.')
        speak('Say hello to get started or say help for available commands.')
        running = True
        while running:
            cmd = listen()
            running = process_command(cmd)
    except KeyboardInterrupt:
        logger.info('Assistant terminated by user')
        speak('System shutting down.')
    except Exception as e:
        logger.critical(f'Critical error: {e}')
        speak('A critical error occurred.')

# Run the assistant
if __name__ == '__main__':
    main()


## (Optional) Cleanup: Disable Colab Widget Manager

Run this cell if you want to disable third-party widget support in Colab.

In [ ]:
# Uncomment to disable widget manager
# from google.colab import output
# output.disable_custom_widget_manager()
